In [1]:
import os

import sys

os.environ["PYSPARK_PYTHON"] = sys.executable

os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession

# Create SparkSession
spark = SparkSession.builder \
        .appName("groupby") \
        .getOrCreate()

sc = spark.sparkContext

In [10]:
'''api_logs = sc.parallelize([
    ("REQ001", 200, 120),
    ("REQ002", 201, 250),
    ("REQ003", 400, 50),
    ("REQ004", 404, 80),
    ("REQ005", 500, 500),
    ("REQ006", 503, 900),
    ("REQ007", 301, 100)
])

(request_id, http_status, response_time_ms)

Group based on HTTP status:
200-299 → SUCCESS
300-399 → REDIRECTION
400-499 → CLIENT_ERROR
500-599 → SERVER_ERROR

Expected: 

SUCCESS
    REQ001
    REQ002

REDIRECTION
    REQ007

CLIENT_ERROR
    REQ003
    REQ004

SERVER_ERROR
    REQ005
    REQ006
'''

api_logs = sc.parallelize([
    ("REQ001", 200, 120),
    ("REQ002", 201, 250),
    ("REQ003", 400, 50),
    ("REQ004", 404, 80),
    ("REQ005", 500, 500),
    ("REQ006", 503, 900),
    ("REQ007", 301, 100)
])

def category(status):
    if 200<= status <=299:
        return "success"
    elif 300<= status <=399:
        return "REDIRECTION"
    elif 400 <= status <=499:
        return "CLIENT_ERROR"
    elif 500 <= status <=599:
        return "SERVER_ERROR"
    else:
        return "unknown"

categorized= api_logs.map(lambda x: (category(x[1]),x[0]))

grouped=categorized.groupByKey().mapValues(list)

result =grouped.collect()

for categori,id in result:
    print(categori)
    for r in id:
        print(" ",r)


success
  REQ001
  REQ002
REDIRECTION
  REQ007
CLIENT_ERROR
  REQ003
  REQ004
SERVER_ERROR
  REQ005
  REQ006


In [8]:
'''api_logs = sc.parallelize([
    ("REQ001", 200, 120),
    ("REQ002", 200, 750),
    ("REQ003", 404, 50),
    ("REQ004", 500, 100),
    ("REQ005", 200, 900),
    ("REQ006", 503, 1200)
])

Use :

HTTP status
+
response time


status >= 500
        → SERVER_FAILURE

status >= 400
        → CLIENT_FAILURE

response_time > 500
        → SLOW_SUCCESS

otherwise
        → HEALTHY


HEALTHY
    REQ001

SLOW_SUCCESS
    REQ002
    REQ005

CLIENT_FAILURE
    REQ003

SERVER_FAILURE
    REQ004
    REQ006'''

api_logs = sc.parallelize([
    ("REQ001", 200, 120),
    ("REQ002", 200, 750),
    ("REQ003", 404, 50),
    ("REQ004", 500, 100),
    ("REQ005", 200, 900),
    ("REQ006", 503, 1200)
])

def status(id,response_time):
    if id>=500:
        return "SERVER_FAILURE"
    elif id>=400:
        return "CLIENT_FAILURE"
    elif response_time>500:
        return "SLOW_SUCCESS"
    else:
        return  "HEALTHY"
    
cat=api_logs.map(lambda x:(status(x[1],x[2]),x[0]))

grouped = cat.groupByKey().mapValues(list)
result=grouped.collect()

for category,id in result:
    print(category)
    for r in id:
        print(" ",r)

HEALTHY
  REQ001
SERVER_FAILURE
  REQ004
  REQ006
SLOW_SUCCESS
  REQ002
  REQ005
CLIENT_FAILURE
  REQ003


In [16]:
'''customers = sc.parallelize([
    ("C101", "Anuj", "anuj@gmail.com", 32),
    ("C102", "", "rahul@gmail.com", 29),
    ("C103", "Amit", None, 35),
    ("C104", "Rohit", "invalid-email", 28),
    ("C105", "Priya", "priya@gmail.com", -5),
    ("C106", "Neha", "neha@gmail.com", 150),
    ("C107", "Vijay", "vijay@gmail.com", 40)
])

(customer_id, name, email, age)


name missing
    → MISSING_NAME

email missing
    → MISSING_EMAIL

email doesn't contain "@"
    → INVALID_EMAIL

age < 0 OR age > 120
    → INVALID_AGE

otherwise
    → VALID


Expected :

MISSING_NAME
    C102

MISSING_EMAIL
    C103

INVALID_EMAIL
    C104

INVALID_AGE
    C105
    C106

VALID
    C101
    C107 '''

customers = sc.parallelize([
    ("C101", "Anuj", "anuj@gmail.com", 32),
    ("C102", "", "rahul@gmail.com", 29),
    ("C103", "Amit", None, 35),
    ("C104", "Rohit", "invalid-email", 28),
    ("C105", "Priya", "priya@gmail.com", -5),
    ("C106", "Neha", "neha@gmail.com", 150),
    ("C107", "Vijay", "vijay@gmail.com", 40)
])

def cut(name,mail,age):
    if name =="":
        return "MISSING_NAME"
    elif mail ==None:
        return "MISSING MAIL"
    elif "@" not in mail:
        return "Invalid mail"
    elif  age < 0 or age > 120:
        return "invalid age"
    else:
        return "valid"

cuto= customers.map(lambda x:(cut(x[1],x[2],x[3]),x[0]))

grouped = cuto.groupByKey().mapValues(list)

result =grouped.collect()

for status,id in result:
    print(status)
    for r in id:
        print(" ",r)

invalid age
  C105
  C106
Invalid mail
  C104
MISSING MAIL
  C103
MISSING_NAME
  C102
valid
  C101
  C107


In [19]:
'''events = sc.parallelize([
    ("E001", "2026-08-09 06:30:00"),
    ("E002", "2026-08-09 10:15:00"),
    ("E003", "2026-08-09 14:45:00"),
    ("E004", "2026-08-09 19:30:00"),
    ("E005", "2026-08-09 23:10:00"),
    ("E006", "2026-08-09 02:30:00")
])


05:00 - 11:59
    → MORNING

12:00 - 16:59
    → AFTERNOON

17:00 - 21:59
    → EVENING

22:00 - 04:59
    → NIGHT '''


from datetime import datetime

events = sc.parallelize([
    ("E001", "2026-08-09 06:30:00"),
    ("E002", "2026-08-09 10:15:00"),
    ("E003", "2026-08-09 14:45:00"),
    ("E004", "2026-08-09 19:30:00"),
    ("E005", "2026-08-09 23:10:00"),
    ("E006", "2026-08-09 02:30:00")
])

# Categorization function
def classify_events(event_id, timestamp):
    hour = int(timestamp.split(" ")[1].split(":")[0])
    if 5 <= hour < 12:
        period = "MORNING"
    elif 12 <= hour < 17:
        period = "AFTERNOON"
    elif 17 <= hour < 22:
        period = "EVENING"
    else:
        period = "NIGHT"
    return period

# Map each event to (category, event_id)
categorized = events.map(lambda x: (classify_events(x[0], x[1]), x[0]))

# Group by category
grouped = categorized.groupByKey().mapValues(list)

# Collect results
result = grouped.collect()

# Pretty print
for category, evts in result:
    print(category)
    for e in evts:
        print("   ", e)

NIGHT
    E005
    E006
AFTERNOON
    E003
MORNING
    E001
    E002
EVENING
    E004


In [17]:
'''paths = sc.parallelize([
    "s3://bucket/sales/year=2026/month=08/day=01/file1.parquet",
    "s3://bucket/sales/year=2026/month=08/day=01/file2.parquet",
    "s3://bucket/sales/year=2026/month=08/day=02/file3.parquet",
    "s3://bucket/sales/year=2026/month=07/day=31/file4.parquet",
    "s3://bucket/sales/year=2025/month=12/day=01/file5.parquet"
])


year + month

Expected :

2026-08
    file1
    file2
    file3

2026-07
    file4

2025-12
    file5'''

paths = sc.parallelize([
    "s3://bucket/sales/year=2026/month=08/day=01/file1.parquet",
    "s3://bucket/sales/year=2026/month=08/day=01/file2.parquet",
    "s3://bucket/sales/year=2026/month=08/day=02/file3.parquet",
    "s3://bucket/sales/year=2026/month=07/day=31/file4.parquet",
    "s3://bucket/sales/year=2025/month=12/day=01/file5.parquet"
])

file= paths.map(lambda x: (
                x.split("/")[-4].split("=")[1]+"-"+ 
                x.split("/")[-3].split("=")[1],
                x.split("/")[-1].split(".")[0]
                ))
grouped=file.groupByKey().mapValues(list)

result=grouped.collect()

for year,files in result:
    print(year)
    for r in files:
        print(" ",r)


2025-12
  file5
2026-08
  file1
  file2
  file3
2026-07
  file4


In [26]:
'''sales = sc.parallelize([
    ("O101", "India", "Electronics", 50000),
    ("O102", "India", "Clothing", 10000),
    ("O103", "USA", "Electronics", 70000),
    ("O104", "India", "Electronics", 30000),
    ("O105", "USA", "Clothing", 20000),
    ("O106", "USA", "Electronics", 90000)
])

(order_id, country, category, amount)


Group using both country AND category.


Expected :

("India", "Electronics")
    O101
    O104

("India", "Clothing")
    O102

("USA", "Electronics")
    O103
    O106

("USA", "Clothing")
    O105 '''

sales = sc.parallelize([
    ("O101", "India", "Electronics", 50000),
    ("O102", "India", "Clothing", 10000),
    ("O103", "USA", "Electronics", 70000),
    ("O104", "India", "Electronics", 30000),
    ("O105", "USA", "Clothing", 20000),
    ("O106", "USA", "Electronics", 90000)
])

sale=sales.map(lambda x:((x[1],x[2]),x[0]))

groupedd=sale.groupByKey().mapValues(list).collect()

for (country,category),id in groupedd:
    print(country,",",category)
    for r in id:
        print(" ",r)

India , Clothing
  O102
India , Electronics
  O101
  O104
USA , Clothing
  O105
USA , Electronics
  O103
  O106


In [27]:
orders = sc.parallelize([
    {
        "order_id": "O101",
        "customer": {"id": "C001", "country": "India"},
        "payment": {"method": "UPI", "amount": 5000}
    },
    {
        "order_id": "O102",
        "customer": {"id": "C002", "country": "USA"},
        "payment": {"method": "CARD", "amount": 80000}
    },
    {
        "order_id": "O103",
        "customer": {"id": "C003", "country": "India"},
        "payment": {"method": "CARD", "amount": 120000}
    },
    {
        "order_id": "O104",
        "customer": {"id": "C004", "country": "USA"},
        "payment": {"method": "UPI", "amount": 2000}
    }
])

# Classification function
def classify_order(order):
    country = order["customer"]["country"]
    method = order["payment"]["method"]
    amount = order["payment"]["amount"]

    if country == "India" and amount >= 100000:
        category = "INDIA_HIGH_VALUE"
    elif country == "India":
        category = "INDIA_NORMAL"
    elif country == "USA" and method == "CARD":
        category = "USA_CARD"
    else:
        category = "USA_OTHER"

    return (category, order["order_id"])

# Map each order to (category, order_id)
categorized = orders.map(classify_order)

# Group by category
grouped = categorized.groupByKey().mapValues(list)

# Collect results
result = grouped.collect()

# Pretty print
for category, order_ids in result:
    print(category)
    for oid in order_ids:
        print("   ", oid)


USA_CARD
    O102
USA_OTHER
    O104
INDIA_NORMAL
    O101
INDIA_HIGH_VALUE
    O103


In [38]:
'''logs = sc.parallelize([
    "2026-08-09 10:15:01 INFO Job started job_id=101",
    "2026-08-09 10:15:10 ERROR Database connection failed",
    "2026-08-09 10:16:00 WARN Memory usage 85%",
    "2026-08-09 10:17:00 ERROR S3 AccessDenied",
    "2026-08-09 10:18:00 INFO Processed 50000 records",
    "2026-08-09 10:19:00 FATAL Executor crashed",
    "INVALID LOG MESSAGE"
])


Create your own parser/grouping function.

INFO
WARNING
ERROR
CRITICAL
INVALID

Rules :

contains " INFO "
      → INFO

contains " WARN "
      → WARNING

contains " ERROR "
      → ERROR

contains " FATAL "
      → CRITICAL

otherwise
      → INVALID'''

logs = sc.parallelize([
    "2026-08-09 10:15:01 INFO Job started job_id=101",
    "2026-08-09 10:15:10 ERROR Database connection failed",
    "2026-08-09 10:16:00 WARN Memory usage 85%",
    "2026-08-09 10:17:00 ERROR S3 AccessDenied",
    "2026-08-09 10:18:00 INFO Processed 50000 records",
    "2026-08-09 10:19:00 FATAL Executor crashed",
    "INVALID LOG MESSAGE"
])

def status1(logg):
    res=logg.split(" ")[2]
    if res=="INFO":
        ps= "INFO"
    elif res=="WARN":
        ps ="Warning"
    elif res=="ERROR":
        ps="ERROR"
    elif res=="FATAL":
        ps ="Critical"
    else:
        ps ="Invaalid"
    
    return res,ps

log=logs.map(status1)
grouped=log.groupByKey().mapValues(list).collect()

for msg,s in grouped:
    print(msg)
    for r in s:
        print(" ",r)


WARN
  Warning
INFO
  INFO
  INFO
ERROR
  ERROR
  ERROR
MESSAGE
  Invaalid
FATAL
  Critical


In [ ]:
'''transactions = [
    ("TXN001", "CUSTOMER_A", 500, "SUCCESS"),
    ("TXN002", "CUSTOMER_B", 700, "FAILED"),
    ("TXN003", "CUSTOMER_A", 300, "SUCCESS"),

    ("TXN004", "CUSTOMER_C", 900, "SUCCESS"),
    ("TXN005", "CUSTOMER_A", 1200, "FAILED"),
    ("TXN006", "CUSTOMER_B", 450, "SUCCESS"),

    ("TXN007", "CUSTOMER_D", 800, "FAILED"),
    ("TXN008", "CUSTOMER_A", 600, "SUCCESS"),
    ("TXN009", "CUSTOMER_E", 1000, "SUCCESS"),

    ("TXN010", "CUSTOMER_A", 1500, "SUCCESS"),
    ("TXN011", "CUSTOMER_C", 350, "FAILED"),
    ("TXN012", "CUSTOMER_A", 750, "SUCCESS")
]

rdd = sc.parallelize(transactions, 4)

# 1. Count Transection per partition : Find the partition index and number of transection 

# 2. Total Amount per partition : Find the partition index and total amount of transection

# 3. Count Sucessful transection per partition : Find the partition index and number of successful transection

# 4. Count Faild Transection per partition : Find the partition index and number of failed transection

# 5. Total Amount of Successful Transection per partition : Find the partition index and total amount of successful transection

# 6. Find the maximum transection amount per partition : Find the partition index and maximum transection amount

# 7. Find the minimum transection amount per partition : Find the partition index and minimum transection amount

# 8. Find the average transection amount per partition : Find the partition index and average transection amount

# 9. Find the total number of unique customers per partition : Find the partition index and total number of unique customers'''

In [ ]:
transactions =sc.parallelize([
    ("TXN001", "CUSTOMER_A", 500, "SUCCESS"),
    ("TXN002", "CUSTOMER_B", 700, "FAILED"),
    ("TXN003", "CUSTOMER_A", 300, "SUCCESS"),

    ("TXN004", "CUSTOMER_C", 900, "SUCCESS"),
    ("TXN005", "CUSTOMER_A", 1200, "FAILED"),
    ("TXN006", "CUSTOMER_B", 450, "SUCCESS"),

    ("TXN007", "CUSTOMER_D", 800, "FAILED"),
    ("TXN008", "CUSTOMER_A", 600, "SUCCESS"),
    ("TXN009", "CUSTOMER_E", 1000, "SUCCESS"),

    ("TXN010", "CUSTOMER_A", 1500, "SUCCESS"),
    ("TXN011", "CUSTOMER_C", 350, "FAILED"),
    ("TXN012", "CUSTOMER_A", 750, "SUCCESS")
],5)

## 1. Count Transection per partition : Find the partition index and number of transection 

def cnt(index,itrator):
    
    count=0
    for record in itrator:
        count+=1
    yield (index,count)

res = transactions.mapPartitionsWithIndex(cnt).collect()
print(res)


[(0, 2, []), (1, 2, []), (2, 2, []), (3, 2, []), (4, 4, [])]


In [43]:
## 2. Total Amount per partition : Find the partition index and total amount of transection

def total(index,amt):
    totall=0
    for records in amt:
        totall+= records[2]
    yield (index,totall)

res =transactions.mapPartitionsWithIndex(total).collect()
print(res)

[(0, 1200), (1, 1200), (2, 1650), (3, 1400), (4, 3600)]


In [49]:
## 3. Count Sucessful transection per partition : Find the partition index and number of successful transection
def stats(index,itration):
    result=0
    for r in itration:
        if r[3]=="SUCCESS":
            result+=1
    yield(index,result)

res=transactions.mapPartitionsWithIndex(stats).collect()
print(res)


[(0, 1), (1, 2), (2, 1), (3, 1), (4, 3)]


In [53]:
## 4. Count Faild Transection per partition : Find the partition index and number of failed transection
def faild(index,itrator):
    count=0
    for r in itrator:
        if r[3]=="FAILED":
            count+=1
    yield (index,count)

res1=transactions.mapPartitionsWithIndex(faild).collect()
print(res1)

[(0, 1), (1, 0), (2, 1), (3, 1), (4, 1)]


In [55]:
## 5. Total Amount of Successful Transection per partition : Find the partition index and total amount of successful transection
def amt(index,itration):
  
    total=0
    for r in itration:
        if r[3]=="SUCCESS":
            
            total+=r[2]
    yield(index,total)

res=transactions.mapPartitionsWithIndex(amt).collect()
print(res)

[(0, 500), (1, 1200), (2, 450), (3, 600), (4, 3250)]


In [ ]:
## 6. Find the maximum transection amount per partition : Find the partition index and maximum transection amount


def maximum(index, iterator):
    amounts = [r[2] for r in iterator]   
    if amounts:                          
        yield (index, max(amounts))
    else:
        yield (index, None)


res = transactions.mapPartitionsWithIndex(maximum).collect()
print(res)


[(0, 700), (1, 900), (2, 1200), (3, 800), (4, 1500)]


In [ ]:
## 7. Find the minimum transection amount per partition : Find the partition index and maximum transection amount


def minimum(index, iterator):
    amounts = [r[2] for r in iterator]   
    if amounts:                          
        yield (index, min(amounts))
    else:
        yield (index, None)

res = transactions.mapPartitionsWithIndex(minimum).collect()
print(res)

[(0, 500), (1, 300), (2, 450), (3, 600), (4, 350)]


In [60]:
## 8. Find the average transection amount per partition : Find the partition index and maximum transection amount

def avg_amt(index, iterator):
    amounts = [r[2] for r in iterator]   
    if amounts:                          
        avg_val = sum(amounts) / len(amounts)
        yield (index, avg_val)
    else:
        yield (index, None)


res = transactions.mapPartitionsWithIndex(avg_amt).collect()
print(res)


[(0, 600.0), (1, 600.0), (2, 825.0), (3, 700.0), (4, 900.0)]


In [61]:
## 9. Find the Total transection amount per partition : Find the partition index and maximum transection amount

def totall(index, iterator):
    amounts = [r[2] for r in iterator]   
    if amounts:                          
        addtion = sum(amounts)
        yield (index, addtion)
    else:
        yield (index, None)


res = transactions.mapPartitionsWithIndex(totall).collect()
print(res)

[(0, 1200), (1, 1200), (2, 1650), (3, 1400), (4, 3600)]
